# Chapter 10: Retrieval Augmented Generation(RAG) and Verifying Hallucinations

While integrating a *Retrieval-Augmented Generation(RAG)* pipeline helps ground LLM responses in real-World facts, systems can still suffer from over-generalization, context misinterpretation, and fabricated facts (hallucination). To prevent a production e-commerce system from delivering false parameters to customers, we must programmatically verify the response.

In this section, we will build a data-driven evaluation process. We calculate a Retrieval Coverage Score to check word token overlap, but we also implement a rule-based constraint check to capture critical mismatches. A classic example is a customer querying about a "5 years warranty" while our retrieved policy strictly limits protection to a "2 years warranty". If the model incorrectly confirms the 5-year coverage, the evaluation system must flag it as a critical mistake.

In [2]:
import pandas as pd #import pandas to organize and display our evaluation
import re #Import standard regular expressions for extracting numerical values from strings

#Step 1: Set up the evaluation data showing a critical e-commerce hallucination scenario
eval_data={
    "Query": "Does this product come with a 5 years warranty?",
    "Retrieved Context": "Amazon Policy: his electronic item includes a 2 years warranty covering factory defects.",
    "Model Reply": "Yes, your product is protected by a 5 years warranty from the date of purchase."
}

#Step 2: Calculate the Retrieval Coverage Score
def calculate_rcs(context, answer):
    #clean up common punctuation and split sentences
    context_tokens=set(context.lower().replace(".","").replace(",","").split())
    answer_tokens=set(answer.lower().replace(".","").replace(",","").split())

    #identify unique words
    word_overlap=context_tokens.intersection(answer_tokens)

    #return the ratio of matching words
    if len(answer_tokens)==0:
        return 0.0
    return round(len(word_overlap)/len(answer_tokens),2)
def verify_hallucination(query, context, answer):
    #Extract all numerical values
    query_nums=re.findall(r'\d+',query)# finds 5
    context_nums=re.findall(r'\d+',context)# finds 2
    answer_nums=re.findall(r'\d+',answer)# finds 5

    #catch a critical mistake: if answer matches a wrong query number
    if "5" in answer_nums and "2" in context_nums and "2" not in answer_nums:
        return "CRITICAL MISTAKE: Hallucination Detected (Warranty year mismatch)"
    return "Passed verification"

#Step 4: Run the calculations on our Amazon support bot
coverage=calculate_rcs(eval_data["Retrieved Context"], eval_data["Model Reply"])
verification_result=verify_hallucination(eval_data["Query"], eval_data["Retrieved Context"], eval_data["Model Reply"])

#Step 5: Construct the token verification table
matrix_data=[
    {"Token": "5 years", "In context": "No (Context specifies 2 years)", "Result": "fabricated Fact/hallucination"},
    {"Token": "2 years", "In context": "Yes", "Result": "Omitted by model reply"},
    {"Token": "Coverage Score", "In context": f"{int(coverage * 100)}% word overlap", "Result": verification_result}
]

# Create a dataframe and display it
df= pd.DataFrame(matrix_data)
print("=== RAG HALLUCINATION EVALUATION MATRIX ===")
print(f"User Query: {eval_data['Query']}")
print(f"Model Reply: {eval_data['Model Reply']}\n")
print(df.to_string(index=False))

=== RAG HALLUCINATION EVALUATION MATRIX ===
User Query: Does this product come with a 5 years warranty?
Model Reply: Yes, your product is protected by a 5 years warranty from the date of purchase.

         Token                     In context                                                            Result
       5 years No (Context specifies 2 years)                                     fabricated Fact/hallucination
       2 years                            Yes                                            Omitted by model reply
Coverage Score               20% word overlap CRITICAL MISTAKE: Hallucination Detected (Warranty year mismatch)
